# E-Waste Component Clustering
**SDG 12.4 | Predictive Analysis Project**  
K-Means clustering on CNN feature embeddings with t-SNE visualization

## imports

In [1]:
import os
import json
import warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import datasets, transforms, models
from torch.cuda.amp import autocast

from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    calinski_harabasz_score, adjusted_rand_score,
    normalized_mutual_info_score, homogeneity_score,
    completeness_score, v_measure_score
)
from tqdm import tqdm

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


## configuration

In [2]:
DATA_DIR    = Path(r"D:\Downloads\MASTER_DATASET")
MODEL_PATH  = Path(r"D:\Github Desktop\ewaste_vit_project\models\classification")
OUTPUT_DIR  = Path(r"D:\Github Desktop\ewaste_vit_project\models\clustering")
GRAPHS_DIR  = OUTPUT_DIR / "graphs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GRAPHS_DIR.mkdir(exist_ok=True)

# load best model architecture from classification results
best_info = json.load(open(MODEL_PATH / "best_model.json"))
BEST_ARCH = best_info["best_arch"]
print(f"using best model: {BEST_ARCH}")

IMG_SIZE    = 224
BATCH_SIZE  = 64          # larger batch fine for inference only
NUM_WORKERS = 4
NUM_CLASSES = 20
N_CLUSTERS  = 3           # high / medium / low hazard

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# ground truth hazard mapping for cluster validation
HAZARD_MAP = {
    "Battery": "HIGH",           "PCB": "HIGH",
    "Mobile": "HIGH",            "Television": "HIGH",
    "light bulbs": "HIGH",       "Laptop": "HIGH",
    "Microwave": "MEDIUM",       "Washing Machine": "MEDIUM",
    "Printer": "MEDIUM",         "Capacitor": "MEDIUM",
    "Integrated-micro-circuit": "MEDIUM",
    "microchip": "MEDIUM",       "microprocessor": "MEDIUM",
    "Keyboard": "LOW",           "Mouse": "LOW",
    "LED": "LOW",                "Resistor": "LOW",
    "semiconductor-diode": "LOW","transistor": "LOW",
    "heat-sink": "LOW",
}
HAZARD_COLORS = {"HIGH": "#d7191c", "MEDIUM": "#fdae61", "LOW": "#1a9641"}
HAZARD_INT    = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}
print("configuration loaded")

using best model: resnet50
configuration loaded


## load dataset — all splits combined for richer embedding space

In [3]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(DATA_DIR / "train", transform=transform)
val_ds   = datasets.ImageFolder(DATA_DIR / "val",   transform=transform)
test_ds  = datasets.ImageFolder(DATA_DIR / "test",  transform=transform)

full_ds = ConcatDataset([train_ds, val_ds, test_ds])
loader  = DataLoader(
    full_ds, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=True
)

CLASS_NAMES = train_ds.classes

# build full label list across all splits
all_labels = (
    [s[1] for s in train_ds.samples] +
    [s[1] for s in val_ds.samples]   +
    [s[1] for s in test_ds.samples]
)
all_labels = np.array(all_labels)

print(f"total images for embedding : {len(full_ds)}")
print(f"classes                    : {CLASS_NAMES}")

total images for embedding : 24273
classes                    : ['Battery', 'Capacitor', 'Integrated-micro-circuit', 'Keyboard', 'LED', 'Laptop', 'Microwave', 'Mobile', 'Mouse', 'PCB', 'Printer', 'Resistor', 'Television', 'Washing Machine', 'heat-sink', 'light bulbs', 'microchip', 'microprocessor', 'semiconductor-diode', 'transistor']


## build feature extractor from best trained model

In [4]:
def build_extractor(arch, num_classes, model_path):
    """load trained classifier and strip the head to get feature embeddings."""

    if arch == "resnet18":
        model = models.resnet18(weights=None)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Linear(in_features, 512), nn.BatchNorm1d(512),
            nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(256, num_classes)
        )
        model.load_state_dict(torch.load(model_path, map_location=device))
        # strip head — output 512-dim pooled features
        extractor = nn.Sequential(*list(model.children())[:-1])

    elif arch == "resnet50":
        model = models.resnet50(weights=None)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Linear(in_features, 512), nn.BatchNorm1d(512),
            nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(256, num_classes)
        )
        model.load_state_dict(torch.load(model_path, map_location=device))
        extractor = nn.Sequential(*list(model.children())[:-1])

    elif arch == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(in_features, 256),
            nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        model.load_state_dict(torch.load(model_path, map_location=device))
        extractor = nn.Sequential(model.features, model.avgpool)

    extractor.eval()
    return extractor.to(device)


best_weights_path = MODEL_PATH / BEST_ARCH / f"{BEST_ARCH}_best.pth"
extractor = build_extractor(BEST_ARCH, NUM_CLASSES, best_weights_path)
print(f"extractor loaded from: {best_weights_path}")

extractor loaded from: D:\Github Desktop\ewaste_vit_project\models\classification\resnet50\resnet50_best.pth


## extract feature embeddings

In [5]:
print("extracting embeddings...")
print("approx 3-8 minutes depending on gpu...")

all_embeddings = []

with torch.no_grad():
    for images, _ in tqdm(loader, desc="extracting"):
        images = images.to(device, non_blocking=True)
        with autocast():
            feats = extractor(images)
        feats = feats.squeeze(-1).squeeze(-1)
        all_embeddings.append(feats.cpu().float().numpy())

all_embeddings = np.vstack(all_embeddings)
print(f"embeddings shape : {all_embeddings.shape}")

# standardize — critical before k-means
scaler     = StandardScaler()
emb_scaled = scaler.fit_transform(all_embeddings)
print("embeddings standardized")

# save for reuse
np.save(OUTPUT_DIR / "embeddings.npy",        all_embeddings)
np.save(OUTPUT_DIR / "embeddings_scaled.npy", emb_scaled)
np.save(OUTPUT_DIR / "labels.npy",            all_labels)
print("embeddings saved")

extracting embeddings...
approx 3-8 minutes depending on gpu...


extracting: 100%|██████████| 380/380 [01:09<00:00,  5.43it/s]


embeddings shape : (24273, 2048)
embeddings standardized
embeddings saved


## dimensionality reduction — pca before k-means

In [6]:
# apply pca to reduce noise and speed up k-means
# retain 95% variance — standard practice before clustering high-dim features
pca = PCA(n_components=0.95, random_state=42)
emb_pca = pca.fit_transform(emb_scaled)

print(f"original dims   : {emb_scaled.shape[1]}")
print(f"pca dims (95%)  : {emb_pca.shape[1]}")
print(f"variance kept   : {pca.explained_variance_ratio_.sum():.4f}")

# scree plot
plt.figure(figsize=(10, 5))
cumvar = np.cumsum(pca.explained_variance_ratio_)
plt.plot(range(1, len(cumvar) + 1), cumvar, "b-", linewidth=1.5)
plt.axhline(y=0.95, color="red", linestyle="--", alpha=0.7, label="95% variance")
plt.xlabel("number of components")
plt.ylabel("cumulative explained variance")
plt.title("pca scree plot", fontsize=12, fontweight="bold")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / "pca_scree.png", dpi=150, bbox_inches="tight")
plt.close()
print("scree plot saved")

np.save(OUTPUT_DIR / "embeddings_pca.npy", emb_pca)

original dims   : 2048
pca dims (95%)  : 1498
variance kept   : 0.9500
scree plot saved


## elbow curve — find optimal k

In [7]:
print("computing elbow curve (k=2 to 12)...")

k_range     = range(2, 13)
inertias    = []
sil_scores  = []
db_scores   = []
ch_scores   = []

for k in tqdm(k_range, desc="elbow"):
    km = MiniBatchKMeans(
        n_clusters=k, random_state=42,
        n_init=10, batch_size=1024, max_iter=300
    )
    labels_k = km.fit_predict(emb_pca)
    inertias.append(km.inertia_)

    sample_size = min(5000, len(emb_pca))
    idx = np.random.choice(len(emb_pca), sample_size, replace=False)

    sil_scores.append(silhouette_score(emb_pca[idx], labels_k[idx]))
    db_scores.append(davies_bouldin_score(emb_pca[idx], labels_k[idx]))
    ch_scores.append(calinski_harabasz_score(emb_pca[idx], labels_k[idx]))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("cluster quality metrics vs k", fontsize=13, fontweight="bold")

axes[0,0].plot(k_range, inertias, "b-o", markersize=5, linewidth=1.5)
axes[0,0].axvline(x=N_CLUSTERS, color="red", linestyle="--", alpha=0.7, label=f"k={N_CLUSTERS}")
axes[0,0].set_title("elbow — inertia", fontsize=11)
axes[0,0].set_xlabel("k")
axes[0,0].set_ylabel("inertia")
axes[0,0].legend()
axes[0,0].grid(alpha=0.3)

axes[0,1].plot(k_range, sil_scores, "g-o", markersize=5, linewidth=1.5)
axes[0,1].axvline(x=N_CLUSTERS, color="red", linestyle="--", alpha=0.7)
axes[0,1].set_title("silhouette score (higher=better)", fontsize=11)
axes[0,1].set_xlabel("k")
axes[0,1].set_ylabel("silhouette")
axes[0,1].grid(alpha=0.3)

axes[1,0].plot(k_range, db_scores, "r-o", markersize=5, linewidth=1.5)
axes[1,0].axvline(x=N_CLUSTERS, color="blue", linestyle="--", alpha=0.7)
axes[1,0].set_title("davies-bouldin score (lower=better)", fontsize=11)
axes[1,0].set_xlabel("k")
axes[1,0].set_ylabel("davies-bouldin")
axes[1,0].grid(alpha=0.3)

axes[1,1].plot(k_range, ch_scores, "m-o", markersize=5, linewidth=1.5)
axes[1,1].axvline(x=N_CLUSTERS, color="red", linestyle="--", alpha=0.7)
axes[1,1].set_title("calinski-harabasz score (higher=better)", fontsize=11)
axes[1,1].set_xlabel("k")
axes[1,1].set_ylabel("calinski-harabasz")
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(GRAPHS_DIR / "elbow_cluster_metrics.png", dpi=150, bbox_inches="tight")
plt.close()
print("elbow curve saved")

computing elbow curve (k=2 to 12)...


elbow: 100%|██████████| 11/11 [00:32<00:00,  2.92s/it]


elbow curve saved


## fit final k-means model

In [8]:
print(f"fitting k-means with k={N_CLUSTERS}...")

kmeans = KMeans(
    n_clusters=N_CLUSTERS, random_state=42,
    n_init=20, max_iter=500, algorithm="lloyd"
)
cluster_labels = kmeans.fit_predict(emb_pca)

# compute comprehensive cluster metrics
sample_size = min(5000, len(emb_pca))
idx = np.random.choice(len(emb_pca), sample_size, replace=False)

sil = silhouette_score(emb_pca[idx], cluster_labels[idx])
dbi = davies_bouldin_score(emb_pca[idx], cluster_labels[idx])
chi = calinski_harabasz_score(emb_pca[idx], cluster_labels[idx])

# compare against ground truth hazard labels for external validation
hazard_true = np.array([HAZARD_INT[HAZARD_MAP[CLASS_NAMES[l]]] for l in all_labels])
ari  = adjusted_rand_score(hazard_true, cluster_labels)
nmi  = normalized_mutual_info_score(hazard_true, cluster_labels)
hom  = homogeneity_score(hazard_true, cluster_labels)
comp = completeness_score(hazard_true, cluster_labels)
vm   = v_measure_score(hazard_true, cluster_labels)

metrics = {
    "silhouette_score"      : round(float(sil),  4),
    "davies_bouldin_index"  : round(float(dbi),  4),
    "calinski_harabasz_index": round(float(chi), 4),
    "adjusted_rand_index"   : round(float(ari),  4),
    "normalized_mutual_info": round(float(nmi),  4),
    "homogeneity"           : round(float(hom),  4),
    "completeness"          : round(float(comp), 4),
    "v_measure"             : round(float(vm),   4),
}

print("\nclustering metrics:")
for k, v in metrics.items():
    print(f"  {k:<30} : {v:.4f}")

with open(OUTPUT_DIR / "clustering_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

np.save(OUTPUT_DIR / "cluster_labels.npy", cluster_labels)
print("\nmetrics saved")

fitting k-means with k=3...

clustering metrics:
  silhouette_score               : 0.1662
  davies_bouldin_index           : 4.0827
  calinski_harabasz_index        : 151.6768
  adjusted_rand_index            : 0.0251
  normalized_mutual_info         : 0.0340
  homogeneity                    : 0.0280
  completeness                   : 0.0434
  v_measure                      : 0.0340

metrics saved


## t-sne visualization

In [10]:
# t-sne on pca-reduced embeddings for speed
# sample 5000 points — representative and fast
N_TSNE = min(5000, len(emb_pca))
idx    = np.random.choice(len(emb_pca), N_TSNE, replace=False)

emb_sample     = emb_pca[idx]
label_sample   = all_labels[idx]
cluster_sample = cluster_labels[idx]
hazard_sample  = np.array([HAZARD_MAP[CLASS_NAMES[l]] for l in label_sample])

print(f"running t-sne on {N_TSNE} samples...")
tsne = TSNE(
    n_components=2, perplexity=40, learning_rate=200,
    max_iter=1500, random_state=42, n_jobs=-1
)
tsne_result = tsne.fit_transform(emb_sample)
print("t-sne complete")

np.save(OUTPUT_DIR / "tsne_result.npy",     tsne_result)
np.save(OUTPUT_DIR / "tsne_idx.npy",        idx)
np.save(OUTPUT_DIR / "tsne_labels.npy",     label_sample)
np.save(OUTPUT_DIR / "tsne_clusters.npy",   cluster_sample)

print("t-sne results saved")

running t-sne on 5000 samples...
t-sne complete
t-sne results saved


## plot t-sne — by class, by hazard, by cluster

In [11]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle("t-sne embedding space — e-waste components", fontsize=14, fontweight="bold")

# plot 1: by true class
colors_20 = plt.cm.tab20(np.linspace(0, 1, NUM_CLASSES))
for i, cls in enumerate(CLASS_NAMES):
    mask = label_sample == i
    if mask.sum() == 0:
        continue
    axes[0].scatter(
        tsne_result[mask, 0], tsne_result[mask, 1],
        c=[colors_20[i]], label=cls, alpha=0.6, s=12
    )
axes[0].set_title("by class (20 categories)", fontsize=11)
axes[0].axis("off")
axes[0].legend(bbox_to_anchor=(1.0, 1.0), loc="upper left",
               fontsize=6, markerscale=2, ncol=1)

# plot 2: by ground truth hazard level
for hazard, color in HAZARD_COLORS.items():
    mask = hazard_sample == hazard
    axes[1].scatter(
        tsne_result[mask, 0], tsne_result[mask, 1],
        c=color, label=f"{hazard} hazard", alpha=0.6, s=12
    )
axes[1].set_title("by ground truth hazard level", fontsize=11)
axes[1].axis("off")
axes[1].legend(fontsize=10, markerscale=2)

# plot 3: by k-means cluster assignment
cluster_colors = ["#e41a1c", "#377eb8", "#4daf4a"]
for c in range(N_CLUSTERS):
    mask = cluster_sample == c
    axes[2].scatter(
        tsne_result[mask, 0], tsne_result[mask, 1],
        c=cluster_colors[c], label=f"cluster {c}", alpha=0.6, s=12
    )
axes[2].set_title(f"by k-means cluster (k={N_CLUSTERS})", fontsize=11)
axes[2].axis("off")
axes[2].legend(fontsize=10, markerscale=2)

plt.tight_layout()
plt.savefig(GRAPHS_DIR / "tsne_three_views.png", dpi=150, bbox_inches="tight")
plt.close()
print("t-sne plot saved")

t-sne plot saved


## cluster composition analysis

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("cluster composition analysis", fontsize=13, fontweight="bold")

# stacked bar — class distribution per cluster
counts_matrix = np.zeros((N_CLUSTERS, NUM_CLASSES))
for c in range(N_CLUSTERS):
    mask = cluster_labels == c
    for i in range(NUM_CLASSES):
        counts_matrix[c, i] = (all_labels[mask] == i).sum()

colors_20 = plt.cm.tab20(np.linspace(0, 1, NUM_CLASSES))
bottom    = np.zeros(N_CLUSTERS)
for i, cls in enumerate(CLASS_NAMES):
    axes[0].bar(
        range(N_CLUSTERS), counts_matrix[:, i],
        bottom=bottom, label=cls, color=colors_20[i]
    )
    bottom += counts_matrix[:, i]
axes[0].set_title("class distribution per cluster", fontsize=11)
axes[0].set_xlabel("cluster id")
axes[0].set_ylabel("number of images")
axes[0].set_xticks(range(N_CLUSTERS))
axes[0].set_xticklabels([f"cluster {c}" for c in range(N_CLUSTERS)])
axes[0].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7)

# hazard purity per cluster
hazard_true  = np.array([HAZARD_MAP[CLASS_NAMES[l]] for l in all_labels])
hazard_matrix = np.zeros((N_CLUSTERS, 3))

for c in range(N_CLUSTERS):
    mask = cluster_labels == c
    for j, h in enumerate(["HIGH", "MEDIUM", "LOW"]):
        hazard_matrix[c, j] = (hazard_true[mask] == h).sum()

hazard_matrix_norm = hazard_matrix / hazard_matrix.sum(axis=1, keepdims=True)
x     = np.arange(N_CLUSTERS)
width = 0.25
for j, (h, col) in enumerate(HAZARD_COLORS.items()):
    axes[1].bar(x + j * width, hazard_matrix_norm[:, j],
                width, label=h, color=col, alpha=0.85)
axes[1].set_title("hazard level purity per cluster", fontsize=11)
axes[1].set_xlabel("cluster id")
axes[1].set_ylabel("proportion")
axes[1].set_xticks(x + width)
axes[1].set_xticklabels([f"cluster {c}" for c in range(N_CLUSTERS)])
axes[1].legend(fontsize=10)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(GRAPHS_DIR / "cluster_composition.png", dpi=150, bbox_inches="tight")
plt.close()

# print composition table
print("cluster composition (top classes per cluster):")
print("-" * 55)
for c in range(N_CLUSTERS):
    mask   = cluster_labels == c
    n      = mask.sum()
    top_cls = sorted(range(NUM_CLASSES), key=lambda i: counts_matrix[c, i], reverse=True)[:5]
    print(f"cluster {c} ({n} samples):")
    for i in top_cls:
        cnt = int(counts_matrix[c, i])
        if cnt > 0:
            hz  = HAZARD_MAP.get(CLASS_NAMES[i], "MEDIUM")
            print(f"  {CLASS_NAMES[i]:<35} {cnt:>5}  [{hz}]")
    print()

cluster composition (top classes per cluster):
-------------------------------------------------------
cluster 0 (7503 samples):
  Integrated-micro-circuit             1618  [MEDIUM]
  microprocessor                        886  [MEDIUM]
  microchip                             866  [MEDIUM]
  heat-sink                             846  [LOW]
  semiconductor-diode                   616  [LOW]

cluster 1 (16306 samples):
  Printer                              1773  [MEDIUM]
  light bulbs                          1552  [HIGH]
  Laptop                               1532  [HIGH]
  Keyboard                             1345  [LOW]
  Microwave                            1239  [MEDIUM]

cluster 2 (464 samples):
  Washing Machine                        56  [MEDIUM]
  Television                             54  [HIGH]
  Mobile                                 48  [HIGH]
  Keyboard                               47  [LOW]
  Microwave                              47  [MEDIUM]



## clustering complete
Outputs saved:
- embeddings: `models/clustering/embeddings*.npy`
- metrics: `models/clustering/clustering_metrics.json`
- graphs: `models/clustering/graphs/`